#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import *

#Reading Data From Bronze Table

In [0]:
df = spark.table("workspace.bronze.erp_loc_info")

In [0]:
df.display()

In [0]:
df.count()

#Some Data Transformations

In [0]:
df = (
    df.withColumn("CID", F.col("CID").cast("string"))
    .withColumn("CNTRY", F.col("CNTRY").cast("string"))
)

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df = df.dropDuplicates(["CID"])

In [0]:
df = df.dropna(subset=["CID"])
df = df.fillna({"CNTRY": "Unknown"})

In [0]:
df.filter(F.col("CID").isNull()).count()
df.filter(F.col("CNTRY").isNull()).count()

#Renaming Columns Name

In [0]:
df = (
    df.withColumnRenamed("CID", "customer_key")
    .withColumnRenamed("CNTRY", "country")
)
df.display()

#Write into Silver Layer

In [0]:
(
    df.write
        .mode("overwrite")
        .option("overwriteSchema","true")
        .format("delta")
        .saveAsTable("silver.erp_loc_info")
)